# Car Price Feature Store — Feature Group & Feature View

## What this notebook does
In `01_Data_Unit_Tests_CarPrice.ipynb` we validated the raw data and confirmed it meets our quality contract. Here we take the next step: **clean the data**, **engineer features**, and **store everything in the Hopsworks Feature Store** so it can be versioned, reused, and reliably retrieved for model training.

This notebook covers the full feature store pipeline in two parts:

**Part 1 — Feature Group Creation** (adapted from `01_Feature_Store_Feature_Group_Creation.ipynb`):

1. Load and clean `train.csv` using the same `clean_car_data` logic from our Kedro pipeline.
2. Engineer new features (`car_age`, `mileage_per_year`, `owner_to_age_ratio`, etc.).
3. Validate the engineered features with Great Expectations before ingestion.
4. Push the validated data to Hopsworks as a single Feature Group: `car_price_features`.

**Part 2 — Feature View & Training Dataset** (adapted from `02_Feature_Store_Feature_View.ipynb`):

5. Connect to the Feature Store and retrieve the `car_price_features` Feature Group.
6. Create a Feature View that selects all features and labels `price` as the target.
7. Materialize a training dataset and pull it back locally as `X_train` / `y_train`.
8. Inspect the final feature set that will feed into model training.

## Why a Feature Store for this project?

Unlike a plain CSV pipeline, the Feature Store gives us:
- **Versioning**: if we re-engineer features next week (e.g. add `mpg_per_engine_size`),
  we create Feature Group v2 without overwriting v1 — the model trained on v1 can still
  be reproduced exactly.
- **Reusability**: the Feature View becomes the single source of truth for training data,
  decoupling the data engineering step from the modelling step.
- **Observability**: Hopsworks automatically computes statistics (histograms, correlations)
  on each Feature Group — useful for spotting drift later in the pipeline (point 5 of the project).

---

## Part 1

### 1. Imports and safety checks

In [1]:
import hopsworks
import pandas as pd
import numpy as np
import logging
import os
import importlib.util
import warnings
from pathlib import Path
from typing import Union
from dotenv import load_dotenv

# Suppress warnings to keep notebook clean
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings("ignore", module="hopsworks")


# Setup Logger
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


# Secure Environment Loading
def load_env_vars(root_dir: Union[str, Path]) -> dict:
    """Load environment variables from .env files. Never hardcode credentials."""
    if isinstance(root_dir, str):
        root_dir = Path(root_dir)
    load_dotenv(dotenv_path=root_dir / ".env.default")
    load_dotenv(dotenv_path=root_dir / ".env", override=True)
    return dict(os.environ)

def get_root_dir(default_value: str = "..") -> Path:
    """Returns the project root directory. Notebooks live in notebooks/, so .. points to car-price-mlops/."""
    return Path(os.getenv("ML_PIPELINE_ROOT_DIR", default_value))

ML_PIPELINE_ROOT_DIR = get_root_dir()
SETTINGS_STORE = load_env_vars(root_dir=ML_PIPELINE_ROOT_DIR)

# Safety check
assert "FS_API_KEY" in SETTINGS_STORE, "Missing FS_API_KEY in .env file!"
assert "FS_PROJECT_NAME" in SETTINGS_STORE, "Missing FS_PROJECT_NAME in .env file!"

logger.info("✅ Environment and Logger setup complete.")

2026-06-24 11:03:52,629 INFO: ✅ Environment and Logger setup complete.


### 2. Data Loading and Cleaning

In [2]:
# Load Raw Data
spec = importlib.util.spec_from_file_location(
    "car_nodes", "../src/car_price_mlops/pipelines/data_feat_engineering/nodes.py"
)
car_nodes = importlib.util.module_from_spec(spec)
spec.loader.exec_module(car_nodes)
clean_car_data = car_nodes.clean_car_data
engineer_features = car_nodes.engineer_features

df_train = pd.read_csv("../data/01_raw/train.csv")
logger.info(f"Raw data loaded: {df_train.shape[0]} rows, {df_train.shape[1]} columns.")

# -------------------------
# Clean & Engineer Features
#--------------------------

# Clean Data:
#   - Typos fixed in the car brands, models, fuel types and transmissions.
#   - Clip/null-out impossible numeric values in age, mileage, tax, etc.
df_clean = clean_car_data(df_train) # from /src/car_price_mlops/pipelines/data_feat_engineering/nodes.py"
logger.info(f"After cleaning: {df_clean.shape[0]} rows (dropped {df_train.shape[0] - df_clean.shape[0]} rows with missing price).")

# Feature Engineering: 
#   - Created car_age, mileage_per_year, is_high_mileage, is_low_paint_quality, owner_to_age_ratio
df_features = engineer_features(df_clean)
logger.info(f"Features engineered. Final shape: {df_features.shape[0]} rows, {df_features.shape[1]} columns.")

df_features.head()

2026-06-24 11:03:52,692 INFO: Raw data loaded: 75973 rows, 14 columns.
2026-06-24 11:03:52,777 INFO: After cleaning: 75973 rows (dropped 0 rows with missing price).
2026-06-24 11:03:52,780 INFO: Features engineered. Final shape: 75973 rows, 21 columns.


,carID,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,...,paintQuality%,previousOwners,hasDamage,car_age,mileage_per_year,is_high_mileage,is_low_paint_quality,owner_to_age_ratio,index,datetime
0,69512,vw,Golf,2016.0,22290,semi-auto,28421.0,petrol,NaN,11.417268,...,63.0,4.0,0.0,10.0,2842.100000,1,0,0.400000,69512,2026-06-24 10:03:52+00:00
1,53000,toyota,Yaris,2019.0,13790,manual,4589.0,petrol,145.0,47.900000,...,50.0,1.0,0.0,7.0,655.571429,0,0,0.142857,53000,2026-06-24 10:03:52+00:00
2,6366,audi,Q2,2019.0,24990,semi-auto,3624.0,petrol,145.0,40.900000,...,56.0,4.0,0.0,7.0,517.714286,0,0,0.571429,6366,2026-06-24 10:03:52+00:00
3,29021,ford,FIESTA,2018.0,12500,manual,9102.0,petrol,145.0,65.700000,...,50.0,NaN,0.0,8.0,1137.750000,0,0,NaN,29021,2026-06-24 10:03:52+00:00
4,10062,bmw,2 Series,2019.0,22995,manual,1000.0,petrol,145.0,42.800000,...,97.0,3.0,0.0,7.0,142.857143,0,0,0.428571,10062,2026-06-24 10:03:52+00:00


### 3. Data Validation with Great Expectations

We already validated the *raw* data in `01_Data_Unit_Tests_CarPrice.ipynb`. Here we run a second, tighter validation pass on the *engineered* features — after `clean_car_data` and `engineer_features` have run — to confirm that the pipeline itself hasn't introduced any problems (e.g. negative `car_age`, null `index`, `price` <= 0) before anything reaches Hopsworks. If this validation fails, ingestion is halted and nothing is pushed to the Feature Store.

In [3]:
import great_expectations as gx
import great_expectations.expectations as gxe
from great_expectations.core.expectation_suite import ExpectationSuite
from great_expectations.core.validation_definition import ValidationDefinition

def build_expectation_suite(context, suite_name: str) -> ExpectationSuite:
    """Builds and registers an Expectation Suite for the car price feature group."""
    try:
        context.suites.delete(suite_name)
    except Exception:
        pass

    suite = context.suites.add(ExpectationSuite(name=suite_name))

    # Identity
    suite.add_expectation(gxe.ExpectColumnValuesToNotBeNull(column="index"))
    suite.add_expectation(gxe.ExpectColumnValuesToBeUnique(column="index"))
    suite.add_expectation(gxe.ExpectColumnValuesToNotBeNull(column="datetime"))

    # Target
    suite.add_expectation(gxe.ExpectColumnValuesToNotBeNull(column="price"))
    suite.add_expectation(gxe.ExpectColumnMinToBeBetween(column="price", min_value=0, strict_min=True))

    # Engineered features
    suite.add_expectation(gxe.ExpectColumnMinToBeBetween(column="car_age", min_value=0, strict_min=False))
    suite.add_expectation(gxe.ExpectColumnMinToBeBetween(column="mileage_per_year", min_value=0, strict_min=False))
    suite.add_expectation(gxe.ExpectColumnValuesToBeBetween(column="owner_to_age_ratio", min_value=0, max_value=None, mostly=0.99))
    suite.add_expectation(gxe.ExpectColumnValuesToBeInSet(column="is_high_mileage", value_set=[0, 1]))
    suite.add_expectation(gxe.ExpectColumnValuesToBeInSet(column="is_low_paint_quality", value_set=[0, 1]))

    # Original columns — renamed
    suite.add_expectation(gxe.ExpectColumnValuesToBeBetween(column="paint_quality_pct", min_value=0, max_value=100, mostly=0.99))
    suite.add_expectation(gxe.ExpectColumnMinToBeBetween(column="previous_owners", min_value=0, strict_min=False))
    suite.add_expectation(gxe.ExpectColumnValuesToBeBetween(column="mileage", min_value=0, max_value=None, mostly=0.99))
    suite.add_expectation(gxe.ExpectColumnValuesToBeBetween(column="engine_size", min_value=0, max_value=8.0, mostly=0.99))
    suite.add_expectation(gxe.ExpectColumnValuesToBeBetween(column="year", min_value=1990, max_value=2026, mostly=0.99))
    suite.add_expectation(gxe.ExpectColumnValuesToBeBetween(column="tax", min_value=0, max_value=None, mostly=0.95))
    suite.add_expectation(gxe.ExpectColumnValuesToBeBetween(column="mpg", min_value=0, max_value=None, mostly=0.95))
    suite.add_expectation(gxe.ExpectColumnValuesToBeInSet(column="has_damage", value_set=[0, 1], mostly=0.95))

    suite.save()
    return suite

### 4. Interacting with the Hopsworks Feature Store

With validation passed, we now push the engineered car price features to Hopsworksas a **Feature Group** (`car_price_features`, version 1). Each row represents one car listing, keyed by `index` (the original `carID`) with `datetime` as the event time. We also enable statistics computation so Hopsworks automatically generates histograms and correlations for every feature.

In [4]:
def to_feature_store(data, group_name, feature_group_version, description, group_description, SETTINGS):
    """Connects to Hopsworks and pushes the engineered DataFrame as a Feature Group."""
    project = hopsworks.login(
        api_key_value=SETTINGS["FS_API_KEY"],
        project=SETTINGS["FS_PROJECT_NAME"]
    )
    fs = project.get_feature_store()
    try:
        fg = fs.get_feature_group("car_price_features", version=1)
        fg.delete()
        print("Deleted broken car_price_features v1.")
    except Exception as e:
        print("Could not get/delete via API:", e)

    fg = fs.get_or_create_feature_group(
        name=group_name,
        version=feature_group_version,
        description=description,
        primary_key=["index"],
        event_time="datetime",
        online_enabled=False,
        stream=True,
    )

    print(f"Uploading {group_name} to Hopsworks...")

    try:
        fg.insert(features=data, overwrite=False, write_options={"wait_for_job": True})
    except Exception as e:
        # Data upload succeeds; the 415 comes from the job-status polling endpoint
        print(f"Insert submitted. Job-status polling raised (non-fatal): {e}")

    for name, desc in group_description.items():
        fg.update_feature_description(name, desc)


    print(f"✅ Successfully ingested {group_name} into Hopsworks!")
    return fg


def validate_and_upload_features(context, df, group_name, suite, feature_descriptions, version=1):
    """Runs GX validation and if passed, pushes to Hopsworks Feature Store."""
    print(f"\n--- Processing {group_name} ---")

    try:
        context.validation_definitions.delete(f"{group_name}_validation")
    except Exception: pass
    try:
        context.data_sources.delete(f"{group_name}_source")
    except Exception: pass

    data_source = context.data_sources.add_pandas(f"{group_name}_source")
    data_asset = data_source.add_dataframe_asset(f"{group_name}_asset")
    batch_definition = data_asset.add_batch_definition_whole_dataframe(f"{group_name}_batch")

    validation_def = ValidationDefinition(
        name=f"{group_name}_validation",
        data=batch_definition,
        suite=suite,
    )
    context.validation_definitions.add(validation_def)

    print("Running Great Expectations Validation...")
    validation_results = validation_def.run(batch_parameters={"dataframe": df})

    if validation_results.success:
        print("✅ Data validation PASSED! Proceeding to Feature Store ingestion.")
        return to_feature_store(
            data=df,
            group_name=group_name,
            feature_group_version=version,
            description=f"Features related to {group_name}.",
            group_description=feature_descriptions,
            SETTINGS=SETTINGS_STORE
        )
    else:
        print("❌ Data validation FAILED! Ingestion halted.")
        for result in validation_results.results:
            if not result.success:
                print(f"   Failed Rule: {result.expectation_config.type}")
                print(f"   Column: {result.expectation_config.kwargs.get('column')}")
        return None


# Initialize GX context and build suite (no validation yet — df_features_hops not created yet)
context = gx.get_context(mode="ephemeral")
suite = build_expectation_suite(context, "car_price_expectations")
print("✅ Suite built successfully!")

2026-06-24 11:03:52,805 INFO: Created temporary directory '/var/folders/f0/h1r1t1l15cz_0gjkxj_2n_qr0000gn/T/tmp4s7ujl5o' for ephemeral docs site
2026-06-24 11:03:52,805 INFO: Loading 'datasources' ->
[]
✅ Suite built successfully!


In [5]:
# Rename columns to be Hopsworks-compatible
df_features_hops = df_features.rename(columns={
    "paintQuality%": "paint_quality_pct",
    "Brand": "brand",
    "model": "model_name",
    "fuelType": "fuel_type",
    "engineSize": "engine_size",
    "previousOwners": "previous_owners",
    "hasDamage": "has_damage",
    "carID": "car_id",
})

# Drop the stray uppercase carID column
df_features_hops = df_features_hops.drop(columns=["carID"], errors="ignore")


# 1. String / categorical columns: Avro needs real None, not float NaN
string_cols = ["brand", "model_name", "transmission", "fuel_type"]
for col in string_cols:
    df_features_hops[col] = (
        df_features_hops[col]
        .astype(object)
        .where(df_features_hops[col].notna(), None)
    )


# 2. Numeric columns: fill remaining NaN with the column median
# (median is always inside each column's valid range, so it never
#  violates the Great Expectations bounds)
numeric_cols = [
    "year", "mileage", "tax", "mpg", "engine_size",
    "paint_quality_pct", "previous_owners",
]
for col in numeric_cols:
    median_val = df_features_hops[col].median()
    df_features_hops[col] = df_features_hops[col].fillna(median_val)

# has_damage is a binary flag —> median could be 0.5, so fill with 0 and cast to int
df_features_hops["has_damage"] = df_features_hops["has_damage"].fillna(0).astype(int)


df_features_hops["is_high_mileage"] = df_features_hops["is_high_mileage"].fillna(0).astype(int)
df_features_hops["is_low_paint_quality"] = df_features_hops["is_low_paint_quality"].fillna(0).astype(int)
df_features_hops["mileage_per_year"] = df_features_hops["mileage_per_year"].fillna(0)
df_features_hops["owner_to_age_ratio"] = df_features_hops["owner_to_age_ratio"].fillna(0)
df_features_hops["car_age"] = df_features_hops["car_age"].fillna(0)

car_feature_descriptions = {
    "index":                "Unique car listing ID (carID).",
    "datetime":             "Timestamp of when this feature row was created.",
    "brand":                "Normalised car brand (e.g. vw, toyota, audi).",
    "model_name":           "Car model name.",
    "year":                 "Year of manufacture.",
    "price":                "Target variable — listed sale price in GBP.",
    "transmission":         "Gearbox type (manual, automatic, semi-auto).",
    "mileage":              "Total mileage on the odometer.",
    "fuel_type":            "Fuel type (petrol, diesel, hybrid, electric).",
    "tax":                  "Annual road tax in GBP.",
    "mpg":                  "Fuel efficiency in miles per gallon.",
    "engine_size":          "Engine displacement in litres.",
    "paint_quality_pct":    "Paint condition score between 0 and 100.",
    "previous_owners":      "Number of previous registered owners.",
    "has_damage":           "Binary flag — 1 if the car has recorded damage.",
    "car_age":              "Derived: current year minus year of manufacture.",
    "mileage_per_year":     "Derived: total mileage divided by car age.",
    "is_high_mileage":      "Derived: 1 if mileage is above the dataset median.",
    "is_low_paint_quality": "Derived: 1 if paint_quality_pct is below 50.",
    "owner_to_age_ratio":   "Derived: number of previous owners per year of age.",
    "car_id":               "Original car listing ID from the source dataset.",
}

# Confirm everything is clean
print("Shape:", df_features_hops.shape)
remaining = df_features_hops.isnull().sum()
print("Nulls remaining:\n", remaining[remaining > 0] if remaining.sum() else "None ✅")

Shape: (75973, 21)
Nulls remaining:
 brand           1521
model_name      1517
transmission    1522
fuel_type       1511
dtype: int64


In [6]:
# Execute digestion

validate_and_upload_features(
    context=context,
    df=df_features_hops,
    group_name="car_price_features",
    suite=suite,
    feature_descriptions=car_feature_descriptions,
    version=1
)


--- Processing car_price_features ---
2026-06-24 11:03:52,874 INFO: No Datasource 'car_price_features_source' to delete
Running Great Expectations Validation...


Calculating Metrics:   0%|          | 0/100 [00:00<?, ?it/s]

✅ Data validation PASSED! Proceeding to Feature Store ingestion.
2026-06-24 11:03:53,153 INFO: Initializing external client
2026-06-24 11:03:53,153 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443


2026-06-24 11:03:54,600 INFO: Python Engine initialized.

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/33934
2026-06-24 11:03:55,954 WARNING: JobWarning: All jobs associated to feature group `car_price_features`, version `1` will be removed.

Deleted broken car_price_features v1.
Uploading car_price_features to Hopsworks...
Feature Group created successfully, explore it at 
https://eu-west.cloud.hopsworks.ai:443/p/33934/fs/23633/fg/44276


Uploading Dataframe: 100.00% |██████████| Rows 75973/75973 | Elapsed Time: 00:01 | Remaining Time: 00:00


Launching job: car_price_features_1_offline_fg_materialization
Insert submitted. Job-status polling raised (non-fatal): Metadata operation error: (url: https://eu-west.cloud.hopsworks.ai/hopsworks-api/api/project/33934/jobs/car_price_features_1_offline_fg_materialization/executions). Server response: 
HTTP code: 415, HTTP reason: Unsupported Media Type, body: b'{"errorCode":120004,"usrMsg":"HTTP 415 Unsupported Media Type","errorMsg":"Web application exception occurred"}', error code: 120004, error msg: Web application exception occurred, user msg: HTTP 415 Unsupported Media Type
✅ Successfully ingested car_price_features into Hopsworks!


## Part 2: Feature View & Training Dataset

With the `car_price_features` Feature Group successfully ingested in Part 1, we now build the  **Feature View** — the component that sits between the Feature Store and the model training  pipeline.

A Feature View is a named, versioned query over one or more Feature Groups. It defines:
- **Which features** go into training (the columns selected from the feature group)
- **Which column is the label** (`price` in our case)
- **The time range** of data to include (we use all available data)

Once the Feature View exists, we materialize a **Training Dataset** — a point-in-time snapshot of the features and labels that gets stored in Hopsworks and can be retrieved locally as `X_train` / `y_train`.


### 5. Connect to the Feature Store and retrieve the Feature Group

We reconnect to Hopsworks and retrieve the `car_price_features` Feature Group (v1) that we ingested in Part 1. This gives us the feature group object we need to build the query for the Feature View.

In [7]:
project = hopsworks.login(
    api_key_value=SETTINGS_STORE["FS_API_KEY"],
    project=SETTINGS_STORE["FS_PROJECT_NAME"],
)
fs = project.get_feature_store()

fg = fs.get_feature_group("car_price_features", version=1)
print(f"✅ Retrieved feature group: {fg.name} v{fg.version}")
print(f"   Features: {[f.name for f in fg.features]}")

2026-06-24 11:39:27,472 INFO: Closing external client and cleaning up certificates.
2026-06-24 11:39:27,475 INFO: Connection closed.
2026-06-24 11:39:27,476 INFO: Initializing external client
2026-06-24 11:39:27,476 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443


2026-06-24 11:39:28,860 INFO: Python Engine initialized.

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/33934
✅ Retrieved feature group: car_price_features v1
   Features: ['car_id', 'brand', 'model_name', 'year', 'price', 'transmission', 'mileage', 'fuel_type', 'tax', 'mpg', 'engine_size', 'paint_quality_pct', 'previous_owners', 'has_damage', 'car_age', 'mileage_per_year', 'is_high_mileage', 'is_low_paint_quality', 'owner_to_age_ratio', 'index', 'datetime']


### 6. Clean up existing Feature Views

Hopsworks' free tier has limits on the number of Feature Views and Training Datasets. Before creating a new one, we delete any existing `car_price_feature_view` versions to avoid hitting those limits. This is safe to run even if no views exist yet.

In [8]:
try:
    existing_views = fs.get_feature_views(name="car_price_feature_view")
    for fv in existing_views:
        try:
            fv.delete_all_training_datasets()
            fv.delete()
            print(f"Deleted old feature view: car_price_feature_view v{fv.version}")
        except Exception as e:
            print(f"Could not delete view v{fv.version}: {e}")
except Exception:
    print("No existing feature views found — nothing to delete.")

### 7. Create the Feature View

We build a query that selects all features from `car_price_features` and create a Feature View named `car_price_feature_view`. The `labels ["price"]` argument tells Hopsworks that `price` is the target variable: it will be separated into `y_train` when we retrieve the training dataset, while all other columns become `X_train`.

In [9]:
query = fg.select_all()

feature_view = fs.create_feature_view(
    name="car_price_feature_view",
    description="All engineered features for car price prediction, with price as the target label.",
    query=query,
    labels=["price"],
)

print(f"✅ Feature View created: {feature_view.name} v{feature_view.version}")

Feature view created successfully, explore it at 
https://eu-west.cloud.hopsworks.ai:443/p/33934/fs/23633/fv/car_price_feature_view/version/1
✅ Feature View created: car_price_feature_view v1


### 8. Materialize the Training Dataset

We materialize a Training Dataset (v1) from the Feature View. This creates a versioned, frozen snapshot of the features and labels in Hopsworks storage. We omit `start_time` and `end_time` to include all available data our entire 75,973 car listings. (**In our case the materialization never started running so we has to do it manually**).

In [13]:
try:
    feature_view.create_training_data(
        version=1,
        description="Car price training dataset v1 — all features, price as label.",
        data_format="csv",
        write_options={"wait_for_job": True},
        coalesce=False,
    )
    print("✅ Training dataset materialized successfully!")
except Exception as e:
    print(f"Training dataset submitted. Job-status polling raised (non-fatal): {e}")

Training dataset submitted. Job-status polling raised (non-fatal): Metadata operation error: (url: https://eu-west.cloud.hopsworks.ai/hopsworks-api/api/project/33934/featurestores/23633/featureview/car_price_feature_view/version/1/trainingdatasets). Server response: 
HTTP code: 500, HTTP reason: Internal Server Error, body: b'{"errorCode":120000,"devMsg":"Cannot invoke \\"io.hops.hopsworks.persistence.entity.datasource.DataSource.setConnector(io.hops.hopsworks.persistence.entity.featurestore.storageconnector.FeaturestoreConnector)\\" because \\"dataSource\\" is null","errorMsg":"A generic error occurred."}', error code: 120000, error msg: A generic error occurred., user msg: 


### 9: Build the training data (free tier workaround)

The Hopsworks Serverless free tier has known limitations on read paths from external clients: the Query Service (Arrow Flight) returns intermittent errors and the training dataset materialization endpoint is unavailable (no offline storage connector). Both of these are infrastructure-level issues on the Hopsworks side, not problems with our data or pipeline.

To unblock model training, we build `X_train` / `y_train` from the in-memory DataFrame produced in Part 1. This is functionally equivalent to what the Feature View would return: the data is the same validated, feature-engineered DataFrame that was ingested into the Feature Group (and is visible in the Hopsworks UI with all 21 registered features).

In a production deployment with a paid Hopsworks tier (or a self-hosted instance with a configured S3/GCS offline connector), the proper flow would be `feature_view.get_training_data()` to retrieve a versioned, frozen snapshot — preserving full reproducibility and decoupling between feature engineering and model training.

In [16]:
# Free tier workaround: build X_train / y_train from the validated DataFrame
# (same data that's registered in the car_price_features Feature Group + Feature View).
print("Building training data from validated DataFrame (Part 1 output)...")

drop_cols = ["index", "datetime", "car_id"]
y_train = df_features_hops[["price"]].copy()
X_train = df_features_hops.drop(columns=["price"] + drop_cols, errors="ignore")

print(f"✅ X_train shape: {X_train.shape}")
print(f"✅ y_train shape: {y_train.shape}")
print(f"\nFeatures ({len(X_train.columns)}):")
for col in X_train.columns:
    print(f"  - {col}")

X_train.head()

Building training data from validated DataFrame (Part 1 output)...
✅ X_train shape: (75973, 17)
✅ y_train shape: (75973, 1)

Features (17):
  - brand
  - model_name
  - year
  - transmission
  - mileage
  - fuel_type
  - tax
  - mpg
  - engine_size
  - paint_quality_pct
  - previous_owners
  - has_damage
  - car_age
  - mileage_per_year
  - is_high_mileage
  - is_low_paint_quality
  - owner_to_age_ratio


,brand,model_name,year,transmission,mileage,fuel_type,tax,mpg,engine_size,paint_quality_pct,previous_owners,has_damage,car_age,mileage_per_year,is_high_mileage,is_low_paint_quality,owner_to_age_ratio
0,vw,Golf,2016.0,semi-auto,28421.0,petrol,145.0,11.417268,2.0,63.0,4.0,0,10.0,2842.100000,1,0,0.400000
1,toyota,Yaris,2019.0,manual,4589.0,petrol,145.0,47.900000,1.5,50.0,1.0,0,7.0,655.571429,0,0,0.142857
2,audi,Q2,2019.0,semi-auto,3624.0,petrol,145.0,40.900000,1.5,56.0,4.0,0,7.0,517.714286,0,0,0.571429
3,ford,FIESTA,2018.0,manual,9102.0,petrol,145.0,65.700000,1.0,50.0,2.0,0,8.0,1137.750000,0,0,0.000000
4,bmw,2 Series,2019.0,manual,1000.0,petrol,145.0,42.800000,1.5,97.0,3.0,0,7.0,142.857143,0,0,0.428571


In [17]:
# Save the Gold layer data to the Kedro model input folder
output_path = "../data/05_model_input/car_price_model_input.parquet"
df_features_hops.to_parquet(output_path, index=False)
print(f"✅ Gold layer saved to {output_path} — shape: {df_features_hops.shape}")

✅ Gold layer saved to ../data/05_model_input/car_price_model_input.parquet — shape: (75973, 21)
